# NB19 — Internal Structure Comparison

**Purpose:** Test whether the resistance-null / constitutive-significant split
observed within metal genes is distinctive, by running the same PGLS at
sub-functional resolution on three comparison categories (AMR, two-component
systems, ABC transporters).

**Pre-specified direction:** None (exploratory). Hypothesis: if the metal split
is specific, other categories should NOT show the same internal contrast.

**Requires:** JupyterHub (Spark for per-genus KO density queries).

**Label:** Exploratory. Run once. No iterative tuning.

**Outputs:**
- `data/internal_structure_results.csv`
- `figures/internal_structure_forest.png`


In [1]:
import sys, re, time
from pathlib import Path
import pandas as pd
import numpy as np
import requests
from statsmodels.stats.multitest import multipletests

PROJECT = Path('/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology')
DATA    = PROJECT / 'data'
FIGS    = PROJECT / 'figures'
TREE_BAC = DATA / 'gtdb_bac_genus_pruned.tree'

MIN_N_GENERA = 100   # consistent with NB18
MIN_KOS      = 5     # min KOs for subcategory to be tested

sys.path.insert(0, str(PROJECT / 'scripts'))
from pgls_utils import run_pgls

_SPARK_AVAILABLE = False
_spark = None
try:
    from berdl_utils import get_spark_session
    _spark = get_spark_session()
    _SPARK_AVAILABLE = True
    print('Spark OK')
except BaseException as _e:
    print(f'Spark unavailable: {_e}')

bac_base  = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
trait_df  = bac_base[['genus_lower', 'mean_levins_B_std']].copy()
print(f'Base data: {len(bac_base)} genera')

# Primary metal KOs to exclude from ABC analysis
gene_df = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
primary_kos = set(gene_df[gene_df['evidence_tier'].isin(['Tier 1','Tier 2'])]['KO'])
print(f'Primary metal KOs: {len(primary_kos)}')

# Metal subcategory reference (existing NB03 results)
if (DATA / '03_category_pgls_results.csv').exists():
    cat_ref = pd.read_csv(DATA / '03_category_pgls_results.csv')
    
    # Add FDR-corrected p-values if not already present
    if 'p_fdr' not in cat_ref.columns:
        from statsmodels.stats.multitest import multipletests
        cat_ref['p_fdr'] = multipletests(cat_ref['p_value'], method='fdr_bh')[1]
    
    print('Metal category reference loaded:')
    # Use the actual column names: 'label' for category, 'beta', 'SE', and now 'p_fdr'
    print(cat_ref[['label', 'beta', 'SE', 'p_fdr']].to_string(index=False))
else:
    print('WARNING: 03_category_pgls_results.csv not found — forest plot metal panel will be skipped')
    cat_ref = None


[berdl_utils] JupyterHub SparkSession acquired: 4.0.1
Spark OK
Base data: 1574 genera
Primary metal KOs: 140
Metal category reference loaded:
          label      beta       SE        p_fdr
F1.1_resistance  0.002523 0.005656 6.556110e-01
 F1.2_transport -0.021781 0.004931 2.755706e-05
   F1.3_sensing -0.018449 0.005443 9.063495e-04
  F1.4_cofactor -0.032736 0.005308 5.167131e-09
F1.5_metabolism -0.020903 0.005257 1.245119e-04


## Block 2 — KEGG BRITE fetch and subcategory definition

In [2]:
# ─── Helper functions ────────────────────────────────────────────────────────

def fetch_brite_b_level(brite_id):
    """Fetch KEGG BRITE file; return {B_level_name: set(KO_ids)}."""
    url = f'https://rest.kegg.jp/get/br:{brite_id}'
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    current_b = None
    subcats = {}
    for line in r.text.splitlines():
        if not line:
            continue
        if line[0] == 'B':
            current_b = line[1:].strip()
        elif line[0] == 'D' and current_b:
            m = re.match(r'\s*(K\d{5})\b', line[1:].strip())
            if m:
                subcats.setdefault(current_b, set()).add(m.group(1))
    return {k: list(v) for k, v in subcats.items()}

def fetch_pathway_ko_defns(map_id):
    """Fetch KEGG pathway ORTHOLOGY section. Returns {KO: definition_str}."""
    r = requests.get(f'https://rest.kegg.jp/get/{map_id}', timeout=30)
    text = r.text
    orth_start = text.find('\nORTHOLOGY')
    orth_end   = text.find('\nCOMPOUND')
    section = text[orth_start:orth_end] if orth_end > 0 else text[orth_start:]
    ko_defns = {}
    for line in section.splitlines():
        m = re.search(r'(K\d{5})\s+(.+)', line)
        if m:
            ko_defns[m.group(1)] = m.group(2).strip()
    return ko_defns


# ─── AMR (ko01504): B-level = antibiotic class; map to mechanism ─────────────
print('Fetching ko01504 (AMR BRITE)...')
amr_b = fetch_brite_b_level('ko01504')
print(f'  {len(amr_b)} B-level entries, {len(set(k for v in amr_b.values() for k in v))} unique KOs')

# Curated mapping: antibiotic-class B-level → mechanism
# Basis: well-established AMR biology:
#   beta-lactamases/aminoglycoside-modifying enzymes/CAT/fosfomycin-lyases = enzymatic
#   efflux modules (RND/MFS/MATE systems) = efflux
#   vancomycin D-Ala modification/PBP changes/ribosomal protection = target mod/protection
AMR_MECH = {
    'beta-Lactamase genes':           'Enzymatic inactivation',
    'Aminoglycoside resistance genes':'Enzymatic inactivation',
    'Phenicol resistance genes':      'Enzymatic inactivation',
    'Fosfomycin resistance genes':    'Enzymatic inactivation',
    'Multipdrug resistance modules':  'Efflux pumps',
    'Tetracycline resistance genes':  'Target modification/protection',
    'Macrolide resistance genes':     'Target modification/protection',
    'Vancomycin resistance modules':  'Target modification/protection',
    'beta-Lactam resistance modules': 'Target modification/protection',
    'CAMP resistance modules':        'Target modification/protection',
    'Trimethoprim resistance genes':  'Target modification/protection',
    'Quinolone resistance genes':     'Target modification/protection',
    'Rifamycin resistance genes':     'Target modification/protection',
    'Sulfonamide resistance genes':   'Target modification/protection',
}
amr_subcats = {}
for b_name, kos in amr_b.items():
    mech = AMR_MECH.get(b_name, 'Other')
    amr_subcats.setdefault(mech, set()).update(kos)
amr_subcats = {k: list(v) for k, v in amr_subcats.items()}
print('\nAMR mechanism subcategories:')
for name, kos in sorted(amr_subcats.items(), key=lambda x: -len(x[1])):
    print(f'  {name}: {len(kos)} KOs')


# ─── Two-component systems (ko02020): classify by definition keyword ──────────
print('\nFetching ko02020 (TCS pathway)...')
tcs_defns = fetch_pathway_ko_defns('ko02020')
print(f'  {len(tcs_defns)} KOs in ORTHOLOGY section')

tcs_hk, tcs_rr, tcs_other = set(), set(), set()
for ko, defn in tcs_defns.items():
    d = defn.lower()
    if 'histidine kinase' in d or 'sensor kinase' in d:
        tcs_hk.add(ko)
    elif 'response regulator' in d:
        tcs_rr.add(ko)
    else:
        tcs_other.add(ko)

tcs_subcats = {}
if len(tcs_hk) >= MIN_KOS:
    tcs_subcats['Sensor histidine kinases'] = list(tcs_hk)
if len(tcs_rr) >= MIN_KOS:
    tcs_subcats['Response regulators'] = list(tcs_rr)
if len(tcs_other) >= MIN_KOS:
    tcs_subcats['Phosphotransfer/other'] = list(tcs_other)
print('TCS subcategories:')
for name, kos in tcs_subcats.items():
    print(f'  {name}: {len(kos)} KOs')


# ─── ABC transporters (ko02010): classify by substrate type ──────────────────
print('\nFetching ko02010 (ABC transporter pathway)...')
abc_defns = fetch_pathway_ko_defns('ko02010')
print(f'  {len(abc_defns)} KOs in ORTHOLOGY section')

SUBST_KW = {
    'Sugar/carbohydrate':  ['sugar','maltose','glucose','ribose','arabinose','galactose',
                            'xylose','trehalose','fructose','mannose','cellobiose','lactose',
                            'oligosaccharide','gluconate','sorbitol','raffinose'],
    'Amino acid/peptide':  ['amino acid','peptide','glutamine','histidine import','serine',
                            'arginine','lysine','methionine','glycine betaine','proline',
                            'leucine','glutamate','aspartate','threonine','cysteine',
                            'phenylalanine','oligopeptide','dipeptide'],
    'Inorganic ion (non-metal)': ['phosphate','sulfate','nitrate','chloride',
                                  'bicarbonate','chromate','molybdate','arsenate',
                                  'tungstate'],
    'Vitamin/cofactor':    ['vitamin','cobalamin','thiamine','riboflavin','folate',
                            'biotin','lipoate','lipoic'],
    'Lipid/LPS':           ['lipopolysaccharide','lipid','fatty acid','phospholipid',
                            'lps','o-antigen','lipoprotein','teichoic'],
    'Drug/multidrug':      ['multidrug','drug','antibiotic','macrolide','tetracycline',
                            'xenobiotic'],
}
abc_subcats = {k: set() for k in SUBST_KW}
abc_unclass = set()
for ko, defn in abc_defns.items():
    if ko in primary_kos:
        continue   # exclude metal transporters from primary list
    d = defn.lower()
    classified = False
    for cat, kws in SUBST_KW.items():
        if any(kw in d for kw in kws):
            abc_subcats[cat].add(ko)
            classified = True
            break
    if not classified:
        abc_unclass.add(ko)

abc_subcats = {k: list(v) for k, v in abc_subcats.items() if len(v) >= MIN_KOS}
if len(abc_unclass) >= MIN_KOS:
    abc_subcats['Other/unclassified'] = list(abc_unclass)
print('ABC transporter substrate subcategories (metal KOs excluded):')
for name, kos in sorted(abc_subcats.items(), key=lambda x: -len(x[1])):
    print(f'  {name}: {len(kos)} KOs')


Fetching ko01504 (AMR BRITE)...


  14 B-level entries, 237 unique KOs

AMR mechanism subcategories:
  Enzymatic inactivation: 106 KOs
  Target modification/protection: 86 KOs
  Efflux pumps: 49 KOs

Fetching ko02020 (TCS pathway)...


  521 KOs in ORTHOLOGY section
TCS subcategories:
  Sensor histidine kinases: 121 KOs
  Response regulators: 112 KOs
  Phosphotransfer/other: 288 KOs

Fetching ko02010 (ABC transporter pathway)...


  515 KOs in ORTHOLOGY section
ABC transporter substrate subcategories (metal KOs excluded):
  Other/unclassified: 248 KOs
  Amino acid/peptide: 87 KOs
  Sugar/carbohydrate: 56 KOs
  Inorganic ion (non-metal): 35 KOs
  Drug/multidrug: 23 KOs
  Lipid/LPS: 16 KOs
  Vitamin/cofactor: 10 KOs


## Block 3 — Spark density computation and PGLS

In [3]:
if not _SPARK_AVAILABLE:
    raise RuntimeError('Spark required for Block 3 — run in JupyterHub')

def density_pgls(ko_list, label, parent):
    """Per-Mb density (Spark) + PGLS. Returns dict or None."""
    ko_prefixed = [f'ko:{k}' for k in ko_list]
    quoted = ', '.join(f"'{k}'" for k in ko_prefixed)
    sql = f"""
        SELECT gm.genome_id,
               regexp_extract(gm.lineage, 'g__([^;]+)', 1) AS genus,
               COUNT(DISTINCT koid.ko)                      AS n_ko,
               gm.length                                    AS genome_length_bp
        FROM kescience_mgnify.genome gm
        JOIN (
            SELECT genome_id, explode(split(kegg_ko, ',')) AS ko
            FROM kescience_mgnify.gene_eggnog
            WHERE kegg_ko IS NOT NULL AND kegg_ko != '-'
        ) koid USING (genome_id)
        WHERE koid.ko IN ({quoted})
        GROUP BY gm.genome_id, gm.lineage, gm.length
    """
    pm = _spark.sql(sql).toPandas()
    pm['genus_lower'] = pm['genus'].str.lower().str.strip()
    pm['ko_per_mb']   = pm['n_ko'] / (pm['genome_length_bp'] / 1e6)
    dens = (pm.groupby('genus_lower', as_index=False)
              .agg(ko_per_mb=('ko_per_mb','mean')))
    merged = trait_df.merge(dens, on='genus_lower', how='inner').copy()
    mu, sd = merged['ko_per_mb'].mean(), merged['ko_per_mb'].std()
    merged['ko_per_mb_z'] = (merged['ko_per_mb'] - mu) / sd
    df_fit = merged.dropna(subset=['ko_per_mb_z','mean_levins_B_std'])
    if len(df_fit) < MIN_N_GENERA:
        print(f'  SKIP {label}: {len(df_fit)} genera < {MIN_N_GENERA}')
        return None
    try:
        res = run_pgls(df_fit, TREE_BAC, response='mean_levins_B_std',
                       predictors=['ko_per_mb_z'], taxon_col='genus_lower',
                       label=label, min_n=MIN_N_GENERA)
        return {'parent_category': parent, 'subcategory': label,
                'n_kos': len(ko_list), 'n_genera': res['n'],
                'lambda_est': res['lambda_est'], 'beta': res['beta'],
                'SE': res['SE'], 't_stat': res['t_stat'],
                'p_raw': res['p_value'], 'partial_R2': res['r2'],
                'delta_aic': res['delta_aic_vs_null']}
    except Exception as exc:
        print(f'  ERROR {label}: {exc}')
        return None


all_results = []

print('=== AMR subcategories ===')
for name, kos in amr_subcats.items():
    if len(kos) < MIN_KOS:
        print(f'  SKIP {name}: {len(kos)} KOs')
        continue
    print(f'  {name} ({len(kos)} KOs)...', end=' ', flush=True)
    r = density_pgls(kos, name, 'AMR')
    if r:
        all_results.append(r)
        print(f'β={r["beta"]:+.4f}, SE={r["SE"]:.4f}, p={r["p_raw"]:.4g}, n={r["n_genera"]}')

print('\n=== Two-component system subcategories ===')
for name, kos in tcs_subcats.items():
    print(f'  {name} ({len(kos)} KOs)...', end=' ', flush=True)
    r = density_pgls(kos, name, 'Two-component systems')
    if r:
        all_results.append(r)
        print(f'β={r["beta"]:+.4f}, SE={r["SE"]:.4f}, p={r["p_raw"]:.4g}, n={r["n_genera"]}')

print('\n=== ABC transporter subcategories ===')
for name, kos in abc_subcats.items():
    print(f'  {name} ({len(kos)} KOs)...', end=' ', flush=True)
    r = density_pgls(kos, name, 'ABC transporters')
    if r:
        all_results.append(r)
        print(f'β={r["beta"]:+.4f}, SE={r["SE"]:.4f}, p={r["p_raw"]:.4g}, n={r["n_genera"]}')

print(f'\nTotal results: {len(all_results)}')


=== AMR subcategories ===
  Enzymatic inactivation (106 KOs)... 

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


β=+0.0058, SE=0.0046, p=0.2091, n=840
  Target modification/protection (86 KOs)... 

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


β=+0.0086, SE=0.0057, p=0.1292, n=1039
  Efflux pumps (49 KOs)... 

...

β=-0.0295, SE=0.0064, p=4.78e-06, n=1007
  Drug/multidrug (23 KOs)... 

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


β=-0.0004, SE=0.0061, p=0.9441, n=918
  Other/unclassified (248 KOs)... 

/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


β=+0.0029, SE=0.0064, p=0.6473, n=1073

Total results: 13


## Block 4 — BH-FDR within category + save results

In [4]:
results_df = pd.DataFrame(all_results)

# BH-FDR within each parent category
fdr_chunks = []
for parent, grp in results_df.groupby('parent_category'):
    ps = grp['p_raw'].values
    _, qs, _, _ = multipletests(ps, method='fdr_bh')
    grp = grp.copy()
    grp['q_bh'] = qs
    grp['sig_fdr'] = qs < 0.05
    fdr_chunks.append(grp)

results_df = pd.concat(fdr_chunks, ignore_index=True)
results_df = results_df.sort_values(['parent_category','beta']).reset_index(drop=True)
results_df.to_csv(DATA / 'internal_structure_results.csv', index=False)
print('Saved: data/internal_structure_results.csv')
print()
print(results_df[['parent_category','subcategory','n_kos','n_genera',
                   'beta','SE','q_bh','sig_fdr']].to_string(index=False))


Saved: data/internal_structure_results.csv

      parent_category                    subcategory  n_kos  n_genera      beta       SE     q_bh  sig_fdr
     ABC transporters                      Lipid/LPS     16      1007 -0.029518 0.006418 0.000033     True
     ABC transporters               Vitamin/cofactor     10       369 -0.014816 0.008205 0.251272    False
     ABC transporters             Amino acid/peptide     87      1033 -0.008217 0.005621 0.336273    False
     ABC transporters      Inorganic ion (non-metal)     35      1057 -0.003175 0.004855 0.798314    False
     ABC transporters                 Drug/multidrug     23       918 -0.000425 0.006050 0.944062    False
     ABC transporters             Sugar/carbohydrate     56       944  0.002068 0.005084 0.798314    False
     ABC transporters             Other/unclassified    248      1073  0.002909 0.006357 0.798314    False
                  AMR         Enzymatic inactivation    106       840  0.005827 0.004636 0.209085   

## Block 5 — Multi-panel forest plot

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

BLUE  = '#0072B2'; ORANGE = '#E69F00'; GREEN = '#009E73'
GREY  = '#999999'; PINK   = '#CC79A7'; RED   = '#D55E00'

# Build metal reference panel from NB03 results
if cat_ref is not None:
    # Try to get beta/SE columns from existing data
    beta_col = 'beta' if 'beta' in cat_ref.columns else None
    se_col   = 'SE'   if 'SE'   in cat_ref.columns else None
    q_col    = 'p_fdr' if 'p_fdr' in cat_ref.columns else None
    
    if beta_col:
        metal_panel = cat_ref.copy()
        metal_panel['subcategory'] = metal_panel.get('category', cat_ref.index)
        metal_panel['sig_fdr'] = metal_panel[q_col] < 0.05 if q_col else True
        metal_panel = metal_panel.sort_values(beta_col)
    else:
        print('WARNING: cannot build metal reference panel — missing columns')
        metal_panel = None
else:
    metal_panel = None

# If metal panel unavailable, use hardcoded reference
if metal_panel is None:
    metal_panel = pd.DataFrame({
        'subcategory': ['Resistance/Detox','Metabolism','Sensing','Transport','Cofactor'],
        'beta':   [+0.003, -0.021, -0.018, -0.022, -0.033],
        'SE':     [ 0.006,  0.005,  0.005,  0.005,  0.005],
        'sig_fdr':[False,   True,   True,   True,   True],
    }).sort_values('beta')

# Panel definitions
panels_data = [
    ('Metal genes\n(NB03 reference)', metal_panel, BLUE, 'beta', 'SE'),
    ('AMR subcategories',
     results_df[results_df.parent_category=='AMR'], ORANGE, 'beta', 'SE'),
    ('Two-component systems',
     results_df[results_df.parent_category=='Two-component systems'], GREEN, 'beta', 'SE'),
    ('ABC transporters',
     results_df[results_df.parent_category=='ABC transporters'], PINK, 'beta', 'SE'),
]

n_panels = len(panels_data)
fig, axes = plt.subplots(1, n_panels, figsize=(4.5 * n_panels, 7), sharey=False)
fig.suptitle(
    'Internal substructure PGLS comparison: metal genes vs. three comparison categories\n'
    'Filled circles = FDR q < 0.05 within category; open = q ≥ 0.05. Blue dashed = P1 β.',
    fontsize=10)

P1_BETA = -0.021
P1_SE   = 0.0037

for ax, (title, df, color, bc, sec) in zip(axes, panels_data):
    df = df.sort_values(bc).reset_index(drop=True)
    ys = range(len(df))
    
    # P1 reference band
    ax.axvspan(P1_BETA - 1.96*P1_SE, P1_BETA + 1.96*P1_SE,
               alpha=0.10, color=BLUE)
    ax.axvline(P1_BETA, color=BLUE, lw=1.0, ls=':', alpha=0.7)
    ax.axvline(0, color='black', lw=0.7, ls='--', alpha=0.5)
    
    for i, row in df.iterrows():
        sig = bool(row.get('sig_fdr', False))
        ax.errorbar(row[bc], i, xerr=1.96*row[sec],
                    fmt='o', color=color,
                    markerfacecolor=color if sig else 'white',
                    markersize=8, markeredgewidth=1.5,
                    elinewidth=1.5, capsize=3, capthick=1.5, zorder=3)
    
    labels = [str(row.get('subcategory', '')).replace(' ', '\n')
              for _, row in df.iterrows()]
    ax.set_yticks(list(ys))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('PGLS β (95% CI)', fontsize=9)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Annotate n_kos if available
    if 'n_kos' in df.columns:
        x_max = ax.get_xlim()[1] if ax.get_xlim()[1] else 0.05
        for i, (_, row) in enumerate(df.iterrows()):
            ax.text(x_max * 0.95, i, f'n={int(row.n_kos)}',
                    va='center', ha='right', fontsize=6, color='#666666',
                    transform=ax.get_yaxis_transform())

plt.tight_layout()
plt.savefig(str(FIGS / 'internal_structure_forest.png'), dpi=300, bbox_inches='tight')
plt.close()
print('Saved: figures/internal_structure_forest.png')


Saved: figures/internal_structure_forest.png


## Block 6 — REPORT.md paragraph draft

In [6]:
# Print the paragraph to add under Finding 4 in REPORT.md

print('\n=== REPORT.md paragraph for Finding 4 ===\n')

# Collect key stats for each comparison category
for parent in ['AMR','Two-component systems','ABC transporters']:
    sub = results_df[results_df.parent_category==parent]
    if sub.empty:
        continue
    betas = sub['beta'].tolist()
    qs    = sub['q_bh'].tolist()
    n_sig = sum(q < 0.05 for q in qs)
    n_tot = len(betas)
    print(f'{parent}: {n_sig}/{n_tot} subcategories FDR-significant; '
          f'β range {min(betas):.3f} to {max(betas):.3f}')

print("""
To confirm that the resistance-null / constitutive-significant split within metal
genes reflects a metal-specific pattern rather than a general feature of functional
gene categories, we tested three comparison categories (AMR, two-component systems,
ABC transporters) at the same sub-functional resolution using the identical PGLS
pipeline (NB19). [PLACEHOLDER — insert actual results after running in JupyterHub.]

The AMR category subdivides into [β values by mechanism]. The two-component system
subcategories (sensor histidine kinases: β=[X]; response regulators: β=[Y]) do not
show the same internal divergence. ABC transporters subdivided by substrate class
show β=[range], with [description of pattern].

In contrast to the metal-gene pattern — where resistance genes (β ≈ +0.003) and
constitutive categories (cofactor biosynthesis β ≈ −0.033) diverge by >0.036 in β
with opposite significance — the comparison categories show [more uniform / also
divergent / convergent] internal structure, [supporting / not supporting] the
specificity of the metal-gene pattern.
""")



=== REPORT.md paragraph for Finding 4 ===

AMR: 1/3 subcategories FDR-significant; β range 0.006 to 0.018
Two-component systems: 0/3 subcategories FDR-significant; β range 0.003 to 0.009
ABC transporters: 1/7 subcategories FDR-significant; β range -0.030 to 0.003

To confirm that the resistance-null / constitutive-significant split within metal
genes reflects a metal-specific pattern rather than a general feature of functional
gene categories, we tested three comparison categories (AMR, two-component systems,
ABC transporters) at the same sub-functional resolution using the identical PGLS
pipeline (NB19). [PLACEHOLDER — insert actual results after running in JupyterHub.]

The AMR category subdivides into [β values by mechanism]. The two-component system
subcategories (sensor histidine kinases: β=[X]; response regulators: β=[Y]) do not
show the same internal divergence. ABC transporters subdivided by substrate class
show β=[range], with [description of pattern].

In contrast to the met